# Finance Scenarios -- 5. Model Validation

Would you put a capital number on these scenarios? A validator asks two things of a scenario generator: does a
run behave like the economy it describes, and would the model have held what actually happened, had it been
fitted in the past on only what was known then? This notebook asks both of the default factor set, with the calibration behind the README,
and shows where the answer is no.

In [1]:
import matplotlib.pyplot as plt

from financescenarios import CalibrationResult, Scenarios, set_display

set_display(decimals=4)
plt.rcParams["figure.dpi"] = 80  # keeps the two six-panel charts below light

## Step 1: the run

The README's calibration of 2026-10-19, saved with the repository, simulated 30 years ahead with 1,000
scenarios, so every scenario gives 30 year-end observations to set against decades of history.

In [2]:
scenarios = Scenarios.from_profiles(
    settings="default",
    factor_set="default",
    api_key="FINANCIAL_MODELING_PREP_KEY",  # replace with your actual API key
    fred_api_key="FRED_KEY",  # optional, replace with your actual FRED key or leave it out
)

# The README's calibration, saved with the repository; scenarios.calibrate() fits a fresh one
calibration = CalibrationResult.load("../docs/assets/readme/calibration.json")
result = scenarios.simulate(calibration=calibration, n_simulations=1000, years=30, seed=42)

## Step 2: do the factors move together the way the economy does?

Each relationship is measured the same way on decades of history and on every scenario.

In [3]:
relationships = scenarios.check_relationships(result)
relationships.select("relationship", "region", "status", "history", "simulated_median")

relationship,region,status,history,simulated_median
str,str,str,f64,f64
"""Policy rate reacts to inflation""","""United States""","""warn""",0.4522,0.2136
"""Policy rate reacts to unemployment""","""United States""","""pass""",-0.5787,-0.3812
"""Rates near neutral at target, higher above it""","""United States""","""pass""",0.0570,0.0336
"""Phillips curve""","""United States""","""pass""",-0.5923,-0.3195
"""Bear markets come with rising unemployment""","""United States""","""warn""",-0.4096,-0.1175
"""Long rate moves with the short rate""","""United States""","""pass""",0.3303,0.5820
"""Inverted curve precedes higher unemployment""","""United States""","""warn""",-0.6712,-0.2604
"""Real equity returns fall with high inflation""","""United States""","""pass""",-0.1525,-0.0804
"""Higher rates slow house prices""","""United States""","""pass""",-0.9628,-0.8992


20 of the 26 relationships pass and 6 warn; none fails. The policy rate answers unemployment in all three
economies (-0.38 against -0.58 in the US), the US Phillips curve slopes the right way (-0.32 against -0.59), the
long rate follows the short rate (0.58 against 0.33) and higher rates slow house prices (-0.90 against -0.96).
What warns is a link that is there but weaker than in history, or stronger: the US policy rate answers inflation
with a slope of 0.21 against history's 0.45, bear markets come with rising unemployment at -0.12 against -0.41,
an inverted curve foreshadows higher unemployment at -0.26 against -0.67 in the US and -0.11 against -0.45 in the
UK, oil passes into US inflation at 0.012 against 0.044, and the euro-area policy rate answers inflation at 0.24,
far more than history's 0.04. The two currency rows pass with the opposite sign to history only because
history's own slope is too uncertain to tell.

## Step 3: does each factor stay within what history has seen?

Each factor's range, largest one-year moves, drawdowns and long stretches, read at year ends against its
longest record; `summary=True` gives one row per factor with what warns or fails.

In [4]:
scenarios.check_realism(result, summary=True)

factor,kind,status,flagged
str,str,str,str
"""united_states_short_rate""","""short rate""","""fail""","""largest one-year fall (fail, tail too thin); longest stretch at the zero lower bound (warn)"""
"""united_states_long_rate""","""long rate""","""warn""","""highest level (warn); largest one-year rise (warn)"""
"""eurozone_short_rate""","""short rate""","""fail""","""highest level (warn, tail too thin); largest one-year rise (fail, tail too thin); largest one-year fall (fail, tail too thin)"""
"""eurozone_long_rate""","""long rate""","""warn""","""highest level (warn, tail too thin); longest curve inversion (warn)"""
"""united_kingdom_short_rate""","""short rate""","""fail""","""longest stretch at the zero lower bound (fail)"""
"""united_kingdom_long_rate""","""long rate""","""fail""","""highest level (fail, tail too thin); largest one-year rise (warn, tail too thin); largest one-year fall (fail, tail too thin); longest curve inversion (fail)"""
"""united_states_inflation""","""inflation""","""fail""","""lowest level (fail, tail too fat); highest level (warn, tail too thin); largest one-year rise (warn)"""
"""eurozone_inflation""","""inflation""","""fail""","""highest level (fail, tail too thin)"""
"""united_kingdom_inflation""","""inflation""","""fail""","""lowest level (warn, tail too fat); highest level (fail, tail too thin); largest one-year rise (fail, tail too thin); largest one-year fall (fail, tail too thin); longest stretch above 5% (fail, tail too thin)"""


No factor passes every test: 5 of the 23 warn and 18 fail. Most flags are tails that are too thin: over 30
years the scenarios reach history's record years less often than history's own windows did, for the rates and
unemployment of all three economies, house prices, Japanese stocks and crude oil's largest one-year rise. A few go
the other way, such as low inflation in the US and the UK ("tail too fat"). No factor reaches a state no economy
can be in.

## Step 4: are the fitted parameters ones the real world could produce?

Guardrails on the fitted parameters themselves: fits at a bound, half-lives, long-run levels against history's
range, volatility against the history behind the fit, and equity crisis regimes against the same model on the
long US record. Only the flagged ones are shown.

In [5]:
scenarios.check_parameters(result, flagged_only=True)

factor,check,parameter,status,value,yardstick,fit,detail
str,str,str,str,f64,str,str,str
"""united_kingdom_inflation""","""volatility""","""volatility""","""warn""",0.5149,"""OECD CPI inflation (United Kingdom), one-year changes at year ends""","""InflationRegimeParams: regime_switching, on OECD history from 1993""","""simulated volatility 0.51x the realised one (0.0146 a year simulated)"""
"""eurozone_unemployment""","""long-run level""","""long_run_mean""","""warn""",0.0716,"""OECD unemployment (France): 7.20% to 10.90%""","""UnemploymentParams: on OECD France data from 2000, linked to eurozone_inflation""","""settles at 7.16%, outside history's 7.20% to 10.90%"""
"""eurozone_short_rate""","""volatility""","""volatility""","""warn""",1.6674,"""its calibration history, step moves""","""PolicyRuleParams: hull_white, on OECD Germany monthly data from 2000, a balanced rule on eurozone_inflation and eurozone_unemployment""","""simulated volatility 1.67x the realised one"""
"""united_kingdom_long_rate""","""volatility""","""volatility""","""warn""",0.6392,"""OECD long-term rate (United Kingdom), one-year changes at year ends""","""TermPremiumParams: hull_white, on OECD history from 1950, the expected path of united_kingdom_short_rate plus a term premium""","""simulated volatility 0.64x the realised one (0.00942 a year simulated)"""
"""japan""","""regime share""","""stationary_distribution[0]""","""warn""",0.4489,"""the same model fitted on the US market's month-end total return since 1926 (Ken French): 0.165""","""RegimeParams: regime_switching, on EWJ monthly data from 2000, 2 regimes, regimes timed on the japan long history, earning the short rate plus a long-run premium""","""long-run share in the crisis regime: 0.449 against 0.165 in the long record, 2.71x"""
"""japan""","""regime duration""","""expected stay in regime 1 (years)""","""warn""",1.1648,"""the same model fitted on the US market's month-end total return since 1926 (Ken French): 3.39""","""RegimeParams: regime_switching, on EWJ monthly data from 2000, 2 regimes, regimes timed on the japan long history, earning the short rate plus a long-run premium""","""expected stay in the calm regime, in years: 1.16 against 3.39 in the long record, 0.34x"""


Six guardrails warn and none fails. UK inflation, fitted on its inflation-targeting era from 1993, swings 0.51
times as much as its longer history; euro-area unemployment settles at 7.16%, just below the 7.20% to 10.90% of
the French history it was fitted on; the euro-area short rate swings 1.67 times its calibration history and the UK
long rate 0.64 times its history; and Japan's equity regimes are furthest from the long US record, in the crisis
regime 44.9% of the time against 16.5%, with calm spells of 1.16 years against 3.39.

## Step 5: back to the end of 2007

Now the harder test. Every series is cut at 31 December 2007, the model is fitted on the 26 years before it,
and 1,000 scenarios run five years ahead from that day's values; the orange line is what happened. The
published values the fit reads besides the data, such as the neutral rate in the policy rule, are the ones
known that day.

In [6]:
factors = [
    "us_broad",
    "united_states_unemployment",
    "united_states_short_rate",
    "united_states_long_rate",
    "united_states_inflation",
    "united_kingdom_inflation",
]
crisis = scenarios.out_of_sample(as_of="2007-12-31", years=5, n_simulations=1000, seed=42, factors=factors)
crisis.plot(factors)

2026-10-19 13:20:27 - financescenarios - WARNING - inflation regimes are fitted on 14 years; the high-inflation regime rests on few episodes (a long history is history_source: oecd without history_start_year)


2026-10-19 13:20:34 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.1315 over 21.9 years; kept the uncorrected speed


<Figure size 1104x512 with 6 Axes>

In [7]:
crisis.table.select("factor", "date", "realized", "p5", "p50", "p95", "pit")

factor,date,realized,p5,p50,p95,pit
str,date,f64,f64,f64,f64,f64
"""united_states_inflation""",2008-12-31,0.0009,0.0127,0.0330,0.0805,0.0070
"""united_states_inflation""",2009-12-31,0.0272,0.0028,0.0270,0.0941,0.5070
"""united_states_inflation""",2010-12-31,0.0150,-0.0019,0.0224,0.0938,0.3240
"""united_states_inflation""",2011-12-31,0.0296,-0.0038,0.0206,0.0978,0.6750
"""united_states_inflation""",2012-12-31,0.0174,-0.0046,0.0196,0.0949,0.4470
"""united_kingdom_inflation""",2008-12-31,0.0308,0.0156,0.0232,0.0297,0.9750
"""united_kingdom_inflation""",2009-12-31,0.0207,0.0133,0.0224,0.0300,0.3990
"""united_kingdom_inflation""",2010-12-31,0.0315,0.0126,0.0215,0.0299,0.9830
"""united_kingdom_inflation""",2011-12-31,0.0360,0.0121,0.0209,0.0295,1.0000


In [8]:
crisis.scores.select("factor", "coverage_50", "coverage_90", "pit_mean", "crps_skill")

factor,coverage_50,coverage_90,pit_mean,crps_skill
str,f64,f64,f64,f64
"""united_states_inflation""",0.8000,0.8000,0.3920,0.5741
"""united_kingdom_inflation""",0.4000,0.4000,0.8128,0.1294
"""united_states_short_rate""",0.0000,0.8000,0.1294,0.4052
"""united_states_long_rate""",0.4000,0.8000,0.1990,0.2176
"""united_states_unemployment""",0.0000,0.4000,0.9574,0.3279
"""us_broad""",0.0000,0.6000,0.0808,-0.5962


The financial crisis falls largely outside the cloud, because of its speed. US unemployment reached 7.3% at
the end of 2008 and 9.9% at the end of 2009, while 1 in 20 scenarios went above 6.36% and 7.45%; only by 2011,
at 8.5%, was it back inside the 90% band. The short rate was cut to 0.11% within the year against a 5th
percentile of 1.05%, and the 10-year rate fell to 2.24% against 2.78%; from 2009 both were back inside the band.
US stocks were down 36.8% by the end of 2008 against a 5th percentile of -12.03%, and still below it a year on.
US inflation dipped below the band at the end of 2008 (0.09% against 1.27%) and came back inside it; UK
inflation, fitted on its calm years since 1993, went above its narrow band in 2008, 2010 and 2011. In the
scores, the 90% band held four of the five years for both rates and US inflation, three for stocks and two for
unemployment and UK inflation, and on CRPS the scenarios lose to a no-change forecast only for stocks.

What did the fit know? Each value the US policy rule read is in the run's notes, with its source:

In [9]:
for note in crisis.notes:
    if note.startswith("united_states_short_rate:"):
        print(note)

united_states_short_rate: the market path's term premium is the New York Fed's ACM estimate as re-estimated today on the whole sample, not the one published on 2007-12-31
united_states_short_rate: EIOPA long-run anchor switched off, the Finance Toolkit has no EIOPA curve by then (its releases start in January 2023)
united_states_short_rate: neutral real rate 2.14% as known on 2007-12-31 (today 1.20%), from Laubach and Williams (2003) model, the New York Fed's real-time estimates, one-sided r* of the vintage's last quarter: 2007Q3 vintage, published 20 December 2007
united_states_short_rate: inflation target 2.00% as known on 2007-12-31 (today 2.00%), from Taylor (1993, p. 202), the rule's own 2% target, as the Federal Reserve stated none before 2012: before 25 January 2012
united_states_short_rate: effective lower bound 0.00% as known on 2007-12-31 (today 0.00%), from Black (1995, p. 1372): with currency to hold, the nominal rate does not go below zero, and no lower policy rate had bee

The rule settles at its neutral real rate plus the 2% target. The fit read 2.14%, the New York Fed's estimate
published on 20 December 2007, where a run today reads 1.20%. Its median short rate still stayed between 2.7%
and 3.0% for all five years, as each run starts on the market's expected path of its day, while the Federal
Reserve held its rate near zero. Estimates of the neutral rate only came down over the following decade; a
backtest that read today's 1.2% would carry into 2007 a lesson learned only afterwards.

## Step 6: back to the end of 2019

The same test from 31 December 2019, so the five years cover the pandemic and the inflation that followed.

In [10]:
pandemic = scenarios.out_of_sample(as_of="2019-12-31", years=5, n_simulations=1000, seed=42, factors=factors)
pandemic.plot(factors)

2026-10-19 13:20:57 - financescenarios - WARNING - inflation regimes are fitted on 26 years; the high-inflation regime rests on few episodes (a long history is history_source: oecd without history_start_year)


2026-10-19 13:21:06 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0494 over 25.9 years; kept the uncorrected speed


2026-10-19 13:21:06 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0787 over 25.9 years; kept the uncorrected speed


2026-10-19 13:21:06 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0209 over 25.9 years; kept the uncorrected speed


2026-10-19 13:21:17 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.1493 over 25.9 years; kept the uncorrected speed


<Figure size 1104x512 with 6 Axes>

In [11]:
pandemic.table.select("factor", "date", "realized", "p5", "p50", "p95", "pit")

factor,date,realized,p5,p50,p95,pit
str,date,f64,f64,f64,f64,f64
"""united_states_inflation""",2020-12-31,0.0136,-0.0015,0.0182,0.0385,0.3520
"""united_states_inflation""",2021-12-31,0.0704,-0.0066,0.0158,0.0407,0.9950
"""united_states_inflation""",2022-12-31,0.0645,-0.0059,0.0162,0.0448,0.9770
"""united_states_inflation""",2023-12-31,0.0335,-0.0078,0.0156,0.0554,0.8500
"""united_states_inflation""",2024-12-31,0.0289,-0.0076,0.0164,0.0579,0.7810
"""united_kingdom_inflation""",2020-12-31,0.0083,0.0077,0.0174,0.0290,0.0660
"""united_kingdom_inflation""",2021-12-31,0.0484,0.0065,0.0183,0.0321,0.9960
"""united_kingdom_inflation""",2022-12-31,0.0924,0.0080,0.0194,0.0330,1.0000
"""united_kingdom_inflation""",2023-12-31,0.0415,0.0063,0.0199,0.0343,0.9860


In [12]:
pandemic.scores.select("factor", "coverage_50", "coverage_90", "pit_mean", "crps_skill")

factor,coverage_50,coverage_90,pit_mean,crps_skill
str,f64,f64,f64,f64
"""united_states_inflation""",0.2000,0.6000,0.7910,0.0750
"""united_kingdom_inflation""",0.0000,0.2000,0.8012,0.2486
"""united_states_short_rate""",0.2000,0.8000,0.4938,0.4495
"""united_states_long_rate""",0.6000,0.8000,0.3666,0.5276
"""united_states_unemployment""",0.4000,0.8000,0.4186,-0.0321
"""us_broad""",0.6000,1.0000,0.7258,0.6911


Here the cloud holds more. US stocks stay inside the 90% band in all five years, through the 2020 crash and the
rally after it. The jump in US unemployment to 6.7% at the end of 2020 is above the band's 5.34%, and the cut to
zero lies below it at first: the short rate at 0.07% at the end of 2020 against a 5th percentile of 0.39%, the
10-year rate at 0.92% against 1.33%; from 2021 both are inside. The inflation that followed does not fit: US
inflation of 7.04% at the end of 2021 and 6.45% at the end of 2022 is above the 95th percentiles of 4.07% and
4.48%, and UK inflation, fitted on its calm years since 1993, sits above its narrow band from 2021 to 2024, at
9.24% in 2022 against 3.30%.

## Step 7: every four years since 2003

One crisis is an anecdote. Five origins four years apart, each run four years ahead, score every year from 2004
to 2023 once, with one row per factor across them.

In [13]:
evaluation = scenarios.out_of_sample_many(
    origins=["2003-12-31", "2007-12-31", "2011-12-31", "2015-12-31", "2019-12-31"],
    years=4,
    n_simulations=1000,
    seed=42,
)
evaluation.scores.select(
    "factor", "origins", "coverage_50", "coverage_90", "bands", "pit_mean", "pit_std", "crps_skill", "tails"
)

2026-10-19 13:21:28 - financescenarios - WARNING - inflation regimes are fitted on 10 years; the high-inflation regime rests on few episodes (a long history is history_source: oecd without history_start_year)


2026-10-19 13:21:34 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.1411 over 25.9 years; kept the uncorrected speed


2026-10-19 13:21:34 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0754 over 25.9 years; kept the uncorrected speed


2026-10-19 13:21:36 - financetoolkit - ERROR - get_government_bond_yield_curve could not be calculated. ValueError: The response for the German government bond yield curve is missing the BBK_SEIS_MATURITY, OBS_VALUE, TIME_PERIOD column(s), which means the source changed its format and the data cannot be interpreted.
Traceback (most recent call last):
  File "/root/finance-scenarios/.venv/lib/python3.11/site-packages/financetoolkit/utilities/error_model.py", line 170, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/root/finance-scenarios/.venv/lib/python3.11/site-packages/financetoolkit/fixedincome/fixedincome_controller.py", line 2311, in get_government_bond_yield_curve
    curve = sources[country]()
            ^^^^^^^^^^^^^^^^^^
  File "/root/finance-scenarios/.venv/lib/python3.11/site-packages/financetoolkit/fixedincome/fixedincome_controller.py", line 2147, in <lambda>
    "Germany": lambda: bundesbank_model.get_yield_curve(start_date, end_date

2026-10-19 13:21:36 - financescenarios - WARNING - interest_rates[eurozone_short_rate]: follows the rule's own path, no market path (No usable data for the Deutsche Bundesbank's zero-coupon curve of Federal securities: an empty table was returned. Public statistics APIs limit how often they can be asked (the OECD allows about 60 downloads an hour), a series may not exist for the requested country, period or window, and a ticker may be mistyped, delisted or briefly unavailable from Yahoo Finance or FMP. Wait a minute and retry, or check the configuration.)


...

2026-10-19 13:23:22 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0494 over 25.9 years; kept the uncorrected speed


2026-10-19 13:23:22 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0787 over 25.9 years; kept the uncorrected speed


2026-10-19 13:23:22 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.0209 over 25.9 years; kept the uncorrected speed


2026-10-19 13:23:33 - financescenarios - WARNING - the finite-sample bias correction of Tang and Chen (2009) would remove all mean reversion from a speed of 0.1493 over 25.9 years; kept the uncorrected speed


factor,origins,coverage_50,coverage_90,bands,pit_mean,pit_std,crps_skill,tails
str,i64,f64,f64,str,f64,f64,f64,str
"""united_states_inflation""",5,0.4000,0.8500,"""ok""",0.6062,0.3033,0.3761,"""low too fat"""
"""eurozone_inflation""",5,0.5500,0.8500,"""ok""",0.5513,0.2768,0.4224,"""ok"""
"""united_kingdom_inflation""",5,0.2000,0.6000,"""ok""",0.6254,0.3629,0.4050,"""high too thin"""
"""united_states_short_rate""",5,0.4000,0.9000,"""ok""",0.4451,0.3004,0.4122,"""low too thin"""
"""united_states_long_rate""",5,0.5500,0.9000,"""ok""",0.3817,0.2417,0.0854,"""low too thin"""
"""eurozone_short_rate""",5,0.2500,0.5000,"""ok""",0.2689,0.3318,0.4340,"""low too thin"""
"""eurozone_long_rate""",5,0.4000,0.6500,"""ok""",0.3343,0.3026,0.0495,"""ok"""
"""united_kingdom_short_rate""",5,0.6000,0.8500,"""ok""",0.5018,0.2766,0.4448,"""low too thin, high too thin"""
"""united_kingdom_long_rate""",5,0.4000,0.8000,"""ok""",0.3314,0.2471,0.1824,"""ok"""


US and euro-area inflation keep 85% of their years inside the 90% band, close to the 0.9 it should hold, and so
do US stocks; UK inflation, fitted on its calm years since 1993, holds 0.60. The US rates hold best, at 0.90 for
both, with the UK short rate at 0.85; the euro-area rates hold least (0.50 for the short rate, 0.65 for the long
one) and, with PIT means of 0.27 and 0.33, ended below the scenarios' middle in most years, as did the US and UK
long rates (0.38 and 0.33). On CRPS every rate beats a no-change forecast, the 10-year rates only just (0.05 to
0.18). Unemployment's bands are too narrow in the US and the euro area (0.50), as are US house prices' (0.50);
the yen's holds 0.70, the euro's is too wide (1.00), and UK unemployment loses to a no-change forecast.

## Recap

- Over 30 years, 20 of the 26 macro relationships pass and 6 warn; the weak links are unemployment's ties to
  stocks and the yield curve, and the policy rates' answer to inflation.
- Against their longest records, every factor warns or fails somewhere, mostly on tails thinner than history's.
- Six parameter guardrails warn, Japan's equity regimes furthest from the long record.
- Fitted on what was known at the end of 2007, the model missed the speed of 2008: unemployment, the first rate
  cuts and the fall in stocks all left the 90% band. Fitted at the end of 2019, stocks stayed inside while the
  2020 jump in unemployment and the 2021-22 inflation surge did not.
- From 2004 to 2023, US and euro-area inflation and US stocks keep about 85% of their years inside their 90%
  bands and the US rates 90%; the euro-area rates, unemployment and house prices are too narrow.

Every check is described in the
[validation documentation](https://www.jeroenbouma.com/projects/financescenarios/docs/validation). Back to the
start: [**0. README Examples**](FinanceScenarios%20-%200.%20README%20Examples.ipynb).